# Notebook 14: Do experimental perturbations resemble the aging signature?

**Goal.** Compare the replicated aging signature with a mitochondrial-stress perturbation (CCCP), using only the coupling genes and a proper random null.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Loads a perturbation table (CCCP versus control, log fold-change per gene). **GENE** and **LFC** are the two column names; in the real GEO file they differ (commonly 'gene', 'log2FoldChange' or 'logFC'), so read the printed names and edit. Then correlates, over the coupling genes only, the aging z-score with the perturbation log fold-change.

*You should see:* 63 coupling genes; rho about 0.26.

In [ ]:
from scipy.stats import spearmanr
from mlc import plots
cccp = pd.read_csv(RAW / "sim" / "cccp_sim.csv.gz")        # real: GSE179611_DMSOLPS_vs_CCCPLPS-ExpDiff.csv.gz
print(cccp.columns.tolist())
GENE, LFC = "Gene", "logFC"                                 # EDIT for real data after reading the printed column names
pert = cccp.set_index(GENE)[LFC]
age = pd.read_csv(RES / "age_signature_meta.csv", index_col=0)
set_genes = [g for g in set(G.MITOPHAGY + G.LYSOSOME) if g in age.index and g in pert.index]
J = pd.DataFrame({"age_z": age.loc[set_genes, "meta_z"], "pert_lfc": pert.loc[set_genes]})
rho, _ = spearmanr(J.age_z, J.pert_lfc); print(len(set_genes), "coupling genes; Spearman rho =", round(rho, 3))

**Cell 3.** **The honest comparison (here borderline: p about 0.06; report it as such, do not round it down).** A transcriptome-wide correlation is easy to get and hard to interpret. Instead, 5,000 random gene sets of the same size give the correlation expected by chance; the empirical p says whether the coupling genes agree with the perturbation more than random genes do. Repeat for every extra perturbation dataset you add.

In [ ]:
rng = np.random.default_rng(SEED); pool = [g for g in age.index if g in pert.index]; null = []
for _ in range(5000):
    r = rng.choice(pool, len(set_genes), replace=False)
    null.append(spearmanr(age.loc[r, "meta_z"], pert.loc[r])[0])
null = np.array(null); p = (1 + (np.abs(null) >= abs(rho)).sum()) / 5001
print("empirical p =", round(p, 4)); plots.fig_perturb(J, rho, p, null); plt.show()

**Cell 4.** Saves, for every gene, whether aging and the perturbation move it in the same direction (1) or not (0); notebook 15 uses it as one evidence layer.

In [ ]:
conc = pd.DataFrame({"pert_concord": (np.sign(age.meta_z.reindex(pert.index)) == np.sign(pert)).astype(float)}).dropna()
conc.to_csv(RES / "perturb_concordance.csv"); print(conc.pert_concord.mean().round(3), "of genes agree in direction")